### 1. Definicja danych (data.dat)

In [ ]:
%%writefile data.dat

# Zbiory
set PRODUCTS := P1 P2;
set FACTORIES := W1 W2;
set WAREHOUSES := M1 M2 M3;
set CUSTOMERS := S1 S2 S3 S4;

# Maksymalna produkcja zakładów
param PRODUCTION: P1  P2 :=
    W1            41  105
    W2            34  116;

# Zapotrzebowanie odbiorców
param DEMAND: P1  P2 :=
    S1        18  38
    S2        12  46
    S3        15  53
    S4        16  51;

# Koszty transportu zakład -> magazyn (identyczne dla obu produktów)
param COST_FACTORY_WAREHOUSE: M1  M2  M3 :=
    W1                         5   2   4
    W2                         7   4   5;

# Koszty transportu magazyn -> odbiorca (identyczne dla obu produktów)
param COST_WAREHOUSE_CUSTOMER: S1  S2  S3  S4 :=
    M1                         15  3   3   4
    M2                         4   10  5   5
    M3                         5   5   5   16;

# Możliwe pojemności magazynów
param Q_M1_1 := 94;
param Q_M1_2 := 109;
param Q_M2_0 := 0;
param Q_M2_1 := 88;
param Q_M2_2 := 109;
param Q_M3_MODULE := 14;

# Koszty operacyjne magazynów (tys. zł)
param K_M1_1 := 348;
param K_M1_2 := 440;
param K_M2_0 := 0;
param K_M2_1 := 324;
param K_M2_2 := 512;
param K_M3_MODULE := 18;

Writing data.dat


### 2. Implementacja modelu (solution.mod)

In [ ]:
%%writefile solution.mod

reset;

# Zbiory
set PRODUCTS;
set FACTORIES;
set WAREHOUSES;
set CUSTOMERS;

# Parametry
param PRODUCTION{FACTORIES, PRODUCTS};
param DEMAND{CUSTOMERS, PRODUCTS};
param COST_FACTORY_WAREHOUSE{FACTORIES, WAREHOUSES};
param COST_WAREHOUSE_CUSTOMER{WAREHOUSES, CUSTOMERS};

# Pojemności i koszty magazynów
param Q_M1_1;
param Q_M1_2;
param Q_M2_0;
param Q_M2_1;
param Q_M2_2;
param Q_M3_MODULE;

param K_M1_1;
param K_M1_2;
param K_M2_0;
param K_M2_1;
param K_M2_2;
param K_M3_MODULE;

# Wczytanie danych
data data.dat;

# Zmienne decyzyjne
var x{FACTORIES, WAREHOUSES, PRODUCTS} >= 0;  # Transport zakład -> magazyn
var y{WAREHOUSES, CUSTOMERS, PRODUCTS} >= 0;  # Transport magazyn -> odbiorca

# Zmienne binarne wyboru konfiguracji magazynów
var u_M1_1 binary;
var u_M1_2 binary;
var u_M2_0 binary;
var u_M2_1 binary;
var u_M2_2 binary;

# Liczba modułów magazynu M3
var n_M3 integer >= 0;

# Ograniczenia produkcyjne
subject to production_constraint{z in FACTORIES, p in PRODUCTS}:
    sum{m in WAREHOUSES} x[z, m, p] <= PRODUCTION[z, p];

# Wybór dokładnie jednej konfiguracji dla M1
subject to warehouse_M1_config:
    u_M1_1 + u_M1_2 = 1;

# Wybór dokładnie jednej konfiguracji dla M2
subject to warehouse_M2_config:
    u_M2_0 + u_M2_1 + u_M2_2 = 1;

# Ograniczenia pojemności magazynu M1
subject to capacity_M1:
    sum{z in FACTORIES, p in PRODUCTS} x[z, 'M1', p] <= Q_M1_1 * u_M1_1 + Q_M1_2 * u_M1_2;

# Ograniczenia pojemności magazynu M2
subject to capacity_M2:
    sum{z in FACTORIES, p in PRODUCTS} x[z, 'M2', p] <= Q_M2_0 * u_M2_0 + Q_M2_1 * u_M2_1 + Q_M2_2 * u_M2_2;

# Ograniczenia pojemności magazynu M3
subject to capacity_M3:
    sum{z in FACTORIES, p in PRODUCTS} x[z, 'M3', p] <= Q_M3_MODULE * n_M3;

# Ograniczenia bilansowe - produkty wpływające = wypływające
subject to flow_balance{m in WAREHOUSES, p in PRODUCTS}:
    sum{z in FACTORIES} x[z, m, p] = sum{s in CUSTOMERS} y[m, s, p];

# Ograniczenia zapotrzebowania
subject to demand_constraint{s in CUSTOMERS, p in PRODUCTS}:
    sum{m in WAREHOUSES} y[m, s, p] = DEMAND[s, p];

# Funkcja celu - minimalizacja całkowitego kosztu
minimize total_cost:
    # Koszty transportu zakład -> magazyn
    sum{z in FACTORIES, m in WAREHOUSES, p in PRODUCTS} COST_FACTORY_WAREHOUSE[z, m] * x[z, m, p]
    # Koszty transportu magazyn -> odbiorca
    + sum{m in WAREHOUSES, s in CUSTOMERS, p in PRODUCTS} COST_WAREHOUSE_CUSTOMER[m, s] * y[m, s, p]
    # Koszty operacyjne magazynów
    + K_M1_1 * u_M1_1 + K_M1_2 * u_M1_2
    + K_M2_0 * u_M2_0 + K_M2_1 * u_M2_1 + K_M2_2 * u_M2_2
    + K_M3_MODULE * n_M3;

Writing solution.mod


### 3. Rozwiązanie modelu

In [ ]:
from amplpy import AMPL

# Inicjalizacja AMPL
ampl = AMPL()

# Ustawienie solvera (highs jest open-source)
ampl.option["solver"] = "cplex"

# Wczytanie modelu
ampl.read("solution.mod")   

# Rozwiązanie
ampl.solve()

# Wyświetlenie wyników
print("=" * 80)
print("WARTOŚĆ FUNKCJI CELU")
print("=" * 80)
print(f"Minimalny całkowity koszt dystrybucji: {ampl.get_objective('total_cost').value():.2f} tys. zł")
print()

print("=" * 80)
print("KONFIGURACJA MAGAZYNÓW")
print("=" * 80)
print(f"Magazyn M1 - opcja 1 (94 jednostek): {ampl.get_variable('u_M1_1').value()}")
print(f"Magazyn M1 - opcja 2 (109 jednostek): {ampl.get_variable('u_M1_2').value()}")
print(f"Magazyn M2 - opcja 0 (nie budowany): {ampl.get_variable('u_M2_0').value()}")
print(f"Magazyn M2 - opcja 1 (88 jednostek): {ampl.get_variable('u_M2_1').value()}")
print(f"Magazyn M2 - opcja 2 (109 jednostek): {ampl.get_variable('u_M2_2').value()}")
print(f"Magazyn M3 - liczba modułów: {ampl.get_variable('n_M3').value()}")
print()

print("=" * 80)
print("TRANSPORT ZAKŁAD -> MAGAZYN")
print("=" * 80)
x_df = ampl.get_variable('x').get_values().to_pandas()
print(x_df[x_df['x.val'] > 0.001])
print()

print("=" * 80)
print("TRANSPORT MAGAZYN -> ODBIORCA")
print("=" * 80)
y_df = ampl.get_variable('y').get_values().to_pandas()
print(y_df[y_df['y.val'] > 0.001])

ModuleNotFoundError: No module named 'amplpy'